# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. You'll learn how to inspect the data structure defined by the Croissant schema, load records, and perform exploratory data analysis (EDA).

### Dataset Source
The dataset source is provided via a Croissant schema URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and make a connection to the records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List the available record sets and their associated field `@id`s for exploration. All references to record sets and fields are made strictly via their Croissant `@id`s, as recommended for robust schema navigation.

In [ ]:
# Extract the available record sets by their @id
record_sets = list(dataset.record_sets.keys())
print("Available Record Sets (@id):")
for i, record_set_id in enumerate(record_sets):
    print(f"  {i+1}. {record_set_id}")
    record_set = dataset.record_sets[record_set_id]
    # List the field @ids in the record set
    field_ids = [f['@id'] for f in record_set.fields]
    print(f"     Fields (@id): {field_ids}")

## 3. Data Extraction

Load data from a specific record set into a pandas DataFrame for further analysis.
For this example, we'll choose the first available record set and its fields.

In [ ]:
# Select the first available record set by @id for demonstration
if record_sets:
    selected_record_set_id = record_sets[0]
    print(f"Loading records from Record Set: {selected_record_set_id}")
else:
    raise ValueError("No record sets found in the dataset.")

# Extract records and load into DataFrame
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print("Available columns (field @id):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing operations using the field `@id`s for maximum reproducibility.

- Filter records based on a numeric field.
- Normalize the field.
- Group by a categorical attribute.

In [ ]:
# Choose a numeric field and a group (categorical) field by @id

numeric_field_id = None
group_field_id = None
# Try to find appropriate field types based on column names (as fallback)
for col in df.columns:
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    elif group_field_id is None and df[col].nunique() < len(df) // 4:
        group_field_id = col

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Filter for values greater than an example threshold
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}")
    display(filtered_df.head())

    # Normalize this numeric column
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Group by a categorical field, if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field was found in the loaded DataFrame. Please check field @id assignments.")

## 5. Visualization

Visualize the distribution of the selected numeric field, or field relationships, using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we have a numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, plot mean value per group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=means.index, y=means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion

In this notebook, you've loaded the FAIR² dataset metadata and records using the `mlcroissant` library, explored record set and field structure using Croissant `@id` attributes, and performed basic exploratory and visualization tasks. For more advanced analytics and modeling, refer to the dataset field documentation and use `mlcroissant` for reproducible data loading.